# 第五章：PyTorch 入门

## 从 NumPy 到 PyTorch——现代深度学习框架的核心抽象

第三章我们用纯 NumPy 实现了 MLP (Multi-Layer Perceptron) 和反向传播，现在引入 PyTorch——它将**自动求导、GPU 加速、
模块化设计**融入工作流，让研究者专注于模型设计而非梯度计算。



## 5.1 Tensor：PyTorch 的核心数据结构

Tensor 就是一个多维数组——和 NumPy 的 ndarray 几乎一样。但 Tensor 多了三个 NumPy 没有的能力：自动求导、GPU 运算、跨设备无缝迁移。

### Tensor 的本质

一个 rank-0 Tensor = 标量。rank-1 = 向量。rank-2 = 矩阵。rank-3 及以上 = 高维数组。每个 Tensor 有三个属性：数据类型（float32/int64/...）、形状（各维度的大小）、设备（CPU 还是 GPU）。

创建 Tensor 和操作 NumPy 数组几乎一样：

```python
x = torch.tensor([1., 2., 3.])      # 从列表创建
x = torch.zeros(3, 4)                # 全零矩阵
x = torch.randn(2, 3)               # 标准正态分布随机初始化
x = torch.from_numpy(np_array)       # 从 NumPy 转换
```

### Tensor vs ndarray 的核心差异

NumPy 的 ndarray 在 CPU 上做数值计算已经足够好。Tensor 的额外价值在于：

**GPU 运算**：`.to('cuda')` 把 Tensor 移到 GPU，之后的所有运算在 GPU 上执行。矩阵乘法在 GPU 上比 CPU 快 10-100 倍——这是深度学习能训练大模型的基础。

**自动求导**：设置 `requires_grad=True` 后，PyTorch 记录对该 Tensor 的所有操作。训练结束时调用 `.backward()` 自动计算梯度。NumPy 没有这个能力。

**内存布局**：Tensor 和 NumPy 共享底层内存——`torch.from_numpy(arr)` 不复制数据，修改 Tensor 会影响原 NumPy 数组。同样，`.numpy()` 也不复制。只有 `.to('cuda')` 会触发数据从 CPU 到 GPU 的传输。

```python
x = torch.tensor([1., 2., 3.], requires_grad=True)  # 要求追踪梯度
y = x.pow(2).sum()        # y = 1² + 2² + 3² = 14
y.backward()              # 自动计算 ∂y/∂x = [2, 4, 6]
print(x.grad)             # tensor([2., 4., 6.])
```


In [ ]:
import os
import torch
import numpy as np

# === 从数据创建 Tensor ===
# 从列表
a = torch.tensor([1, 2, 3, 4, 5])  # 从数据创建张量
print(f"从列表: {a}, dtype={a.dtype}")

# 从 NumPy
b = torch.from_numpy(np.array([1.0, 2.0, 3.0]))  # 创建 NumPy 数组
print(f"从NumPy: {b}")

# 特殊矩阵
zeros = torch.zeros(3, 4)       # 全零
ones = torch.ones(2, 3)          # 全一
rand = torch.randn(3, 3)         # 标准正态分布
eye = torch.eye(3)               # 单位矩阵
arange = torch.arange(0, 10, 2)  # 类似 range

print(f"zeros shape: {zeros.shape}")
print(f"rand: \n{rand}")

# 类型转换
c = torch.tensor([1, 2, 3], dtype=torch.float32)  # 从数据创建张量
print(f"float32: {c}, dtype={c.dtype}")
c_long = c.long()  # 转为 int64
print(f"long: {c_long}, dtype={c_long.dtype}")


In [ ]:
# === Tensor 运算 ===
import torch
x = torch.tensor([[1.0, 2.0], [3.0, 5.0]])  # 从数据创建张量
y = torch.tensor([[5.0, 6.0], [7.0, 8.0]])  # 从数据创建张量

# 逐元素运算
print("x + y =", x + y)
print("x * y =", x * y)  # 逐元素乘（不是矩阵乘！）
print("x ** 2 =", x ** 2)

# 矩阵运算
print("x @ y =\n", x @ y)           # 矩阵乘法
print("x.matmul(y) =\n", x.matmul(y))

# 形状操作
z = torch.arange(12).reshape(3, 4)  # 改变张量形状
print(f"reshape:\n{z}")
print(f"转置:\n{z.T}")
print(f"view:\n{z.view(4, 3)}")  # 改变张量形状（不拷贝数据）
print(f"permute:\n{z.permute(1, 0)}")  # 维度重排

# 归约操作
print(f"sum: {z.sum()}")  # 沿指定维度求和
print(f"mean dim=0: {z.float().mean(dim=0)}")  # 按行平均
print(f"max dim=1: {z.float().max(dim=1)}")    # 按列找最大

# 索引与切片（与 NumPy 相同）
print(f"第一行: {z[0]}")
print(f"前两行: {z[:2]}")
print(f"第二列: {z[:, 1]}")


In [ ]:
# === GPU 支持 ===
import torch
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 数量: {torch.cuda.device_count()}")
    print(f"当前 GPU: {torch.cuda.get_device_name(0)}")
    
    # 数据转移到 GPU
    x_gpu = x.to('cuda')
    print(f"x 在 GPU 上: {x_gpu.device}")
    
    # 在 GPU 上运算
    y_gpu = y.to('cuda')
    result = x_gpu @ y_gpu
    print(f"GPU 计算结果:\n{result}")
    
    # 转回 CPU
    result_cpu = result.cpu()  # 移到 CPU
    print(f"回到 CPU: {result_cpu.device}")
else:
    print("未检测到 GPU，使用 CPU 模式")


## 5.2 Autograd：自动求导引擎

PyTorch 的 **autograd** 系统自动记录所有 Tensor 操作，构建**动态计算图**，然后自动计算
任意 Tensor 对任意参数的梯度。这是 PyTorch 区别于 NumPy 的核心特性。

### 工作原理

1. 设置 `requires_grad=True` 标记需要梯度的 Tensor
2. 前向计算时，autograd 记录所有操作
3. 调用 `.backward()` 自动计算梯度
4. 梯度存储在 `.grad` 属性中

### 一个简单的例子

$$

f(x) = x^3 + 2x^2 + x \quad \Rightarrow \quad f'(x) = 3x^2 + 4x + 1

$$

在 $x=2$ 处：$f'(2) = 12 + 8 + 1 = 21$


### Autograd 的核心机制：动态计算图

Autograd 不是黑魔法。它的工作方式可以分解为三个步骤：

**1. 构建计算图**

每次前向计算时，PyTorch 在后台为每个 Tensor 操作创建一个节点。例如计算 $L = (wx + b - y)^2$：

```
    x ---→ [×] ---→ [+] ---→ [-] ---→ [²] ---→ L
           ↑        ↑        ↑
           w        b        y
```

每个节点记录：(a) 创建它的操作类型（乘法、加法、平方），(b) 输入 Tensor 的引用，(c) 反向传播所需的中间值（如 `ctx.save_for_backward`）。这个图是**动态**的——每次 `forward()` 调用都会重新构建，因此可以处理变长输入和控制流（if/for），这是 PyTorch 区别于 TensorFlow 1.x 静态图的核心优势。

**2. 反向传播：拓扑排序 + 链式法则**

调用 `L.backward()` 时，Autograd 对计算图做**逆拓扑排序**——从输出节点 $L$ 开始，按依赖关系的逆序访问每个节点。对每个节点，调用其 `backward()` 方法计算局部梯度，乘以从下游传来的梯度（链式法则），传递给上游节点。

以上述计算图为例：

$$\frac{\partial L}{\partial L} = 1 \quad \text{(起始梯度)}$$

$$[²]\text{ 节点: }\frac{\partial L}{\partial (wx+b-y)} = 2(wx+b-y) \quad \text{(平方的导数)}$$

$$[-]\text{ 节点: }\frac{\partial L}{\partial (wx+b)} = 2(wx+b-y) \cdot 1 \quad \text{(减法的导数，对减数传 -1)}$$

以此类推，直到每个叶子 Tensor 都收到梯度。

**3. 梯度累加 vs 清零**

`.backward()` 将计算出的梯度**累加**到 `.grad` 属性上，而非覆盖。原因是某些场景需要累积多个子损失的梯度（如 RNN 的时间步展开、GAN 的分别优化），但大多数情况下需要先清零：

```python
optimizer.zero_grad()  # 清零所有参数的 .grad
loss.backward()        # 计算并累加梯度
optimizer.step()       # 使用 .grad 更新参数
```

如果忘记 `zero_grad()`，梯度会累加多轮——导致参数更新步长越来越大，训练发散。

**leaf tensor vs non-leaf tensor**

只有 `requires_grad=True` 且由用户直接创建的 Tensor（非运算结果）才是**叶子节点**，其 `.grad` 会被保留。中间结果的梯度在反向传播后立即释放以节省显存——若需要保留，使用 `.retain_grad()`。


### 什么是张量 (Tensor)

张量 = 多维数组 + 自动求导 + GPU。标量到向量到矩阵的推广：

$$

\text{标量 } s \in \mathbb{R}
\quad \text{向量 } \mathbf{v} \in \mathbb{R}^n
\quad \text{矩阵 } \mathbf{M} \in \mathbb{R}^{m \times n}
\quad \text{三阶 } \mathcal{T} \in \mathbb{R}^{b \times m \times n}

$$

深度学习中：单张图 [3, 224, 224]，一个 batch [64, 3, 224, 224]，一个句子 [128, 768]。


In [ ]:
# === Autograd 基础 ===
import torch
x = torch.tensor(2.0, requires_grad=True)  # 从数据创建张量

# 前向计算
y = x ** 3 + 2 * x ** 2 + x  # f(x) = x^3 + 2x^2 + x
print(f"f(2) = {y}")

# 反向传播
y.backward()  # 反向传播，计算梯度
print(f"f'(2) = {x.grad}")  # 应该是 21
print(f"手动计算: 3*2^2 + 4*2 + 1 = {3*4 + 4*2 + 1}")

# 复杂例子：多变量函数
a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)  # 从数据创建张量
b = torch.tensor([5.0, 5.0, 6.0], requires_grad=True)  # 从数据创建张量

f = (a * b).sum()  # f = sum(a_i * b_i)
f.backward()  # 反向传播，计算梯度
print(f"\na.grad = {a.grad}")  # ∂f/∂a_i = b_i
print(f"b.grad = {b.grad}")    # ∂f/∂b_i = a_i


In [ ]:
# === 梯度下降 with Autograd ===
# 目标：找到 f(x) = (x-3)^2 的最小值（显然 x=3）
import torch
x = torch.tensor(0.0, requires_grad=True)  # 从数据创建张量
lr = 0.1
history = []

for step in range(30):
    loss = (x - 3) ** 2  # f(x) = (x-3)^2
    
    loss.backward()  # 反向传播，计算梯度
    
    with torch.no_grad():  # 更新时不记录梯度
        x -= lr * x.grad
        x.grad.zero_()     # 清空梯度（否则会累加！）
    
    history.append((step, x.item(), loss.item()))  # 取出单个数值

for s, xv, lv in history[::5]:
    print(f"Step {s:2d}: x={xv:.4f}, loss={lv:.6f}")

print(f"\n最终: x={x.item():.6f} (真实答案: 3.0)")  # 取出单个数值

# 画收敛曲线
import matplotlib.pyplot as plt
steps, xs, losses = zip(*history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))  # 创建子图网格
ax1.plot(steps, xs); ax1.axhline(3, color='r', linestyle='--')
ax1.set_title('x'); ax1.set_xlabel('Step'); ax1.grid(True)
ax2.plot(steps, losses); ax2.set_title('Loss')
ax2.set_xlabel('Step'); ax2.grid(True)
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/autograd_gd.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


### 梯度清零：为什么必须放在 `backward()` 之前

把 `zero_grad()` 放在 `backward()` 之后的写法也能运行，但逻辑上有陷阱：

```python
loss.backward()         # 计算本轮梯度
optimizer.step()        # 更新参数
optimizer.zero_grad()   # 清零梯度（为下一轮准备）
```

这个顺序在单步训练中没问题。但当你在 `loss.backward()` 之前还有其他计算（如 `loss1.backward()` + `loss2.backward()` 的梯度累加场景）时，上一轮残留的梯度会污染当前计算。显式放在 `backward()` 前面确保本轮梯度从零开始累加：

```python
optimizer.zero_grad()   # 确保从零开始
loss1.backward()        # 累加损失 1 的梯度
loss2.backward()        # 累加损失 2 的梯度
optimizer.step()        # 用总梯度更新
```

梯度累积的实用场景：GPU 显存装不下大 batch 时，将 batch 拆为多次小 forward，梯度累加后一次性更新——模拟大 batch 的效果而无需大显存。


## 5.3 nn.Module：所有神经网络组件的基类

第三章用纯 NumPy 手动管理每一层的权重矩阵和梯度。当网络有几十层时，手动追踪所有参数不现实。`nn.Module` 自动化了这个管理。

### Module 的核心机制

任何 PyTorch 模型都继承 `nn.Module`。它只要求实现一个方法：

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)  # 定义子模块

    def forward(self, x):
        x = self.fc1(x).relu()
        return x
```

一旦 `nn.Linear` 被赋值给 `self.fc1`，Module 自动将它注册到内部的 `_modules` 字典中。后续调用 `model.parameters()` 会递归遍历所有子模块，收集所有需要训练的参数——不需要手动维护参数列表。

### 为什么 __init__ 和 forward 必须分离

`__init__` 里定义**有哪些参数**——Linear 层的权重矩阵和偏置向量。`forward` 里定义**这些参数怎么用**——先矩阵乘法再激活。

分离的原因是 Module 需要知道参数的存在时机。`__init__` 执行完后，所有子模块和参数已经注册完毕，框架可以：
- 用 `model.to('cuda')` 一次性把所有参数移到 GPU
- 用 `model.parameters()` 一次性收集所有参数给优化器
- 用 `model.state_dict()` 一次性保存所有参数到文件

如果参数在 `forward` 里创建（如局部变量），框架感知不到它们。

### 预置的层：不需要自己写矩阵乘法

PyTorch 内置了所有常见层：

| 层 | 对应操作 | 可学习参数 |
|---|---------|-----------|
| `nn.Linear(in, out)` | $\mathbf{y} = W\mathbf{x} + \mathbf{b}$ | $W, \mathbf{b}$ |
| `nn.Conv2d(in, out, k)` | 2D 卷积 | 卷积核权重 + 偏置 |
| `nn.BatchNorm2d(n)` | 批归一化 | $\gamma, \beta$ |
| `nn.Dropout(p)` | 随机置零 | 无 |
| `nn.ReLU()` | $\max(0, x)$ | 无 |

使用内置层比自己写矩阵乘法有三个好处：参数被自动注册、权重使用正确的初始化策略（如 Kaiming 初始化）、计算使用优化的底层实现（比纯 Python 快几十倍）。


### nn.Module 的设计哲学：为什么 `__init__` 和 `forward` 分离

所有 PyTorch 模型都继承 `nn.Module`，必须实现两个方法：

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)  # 定义子模块（参数在这里）

    def forward(self, x):                # 定义前向计算逻辑
        return self.linear(x).relu()
```

`__init__` 中定义的 `nn.Module` 子对象被自动注册到模型的 `_modules` 字典中。调用 `model.parameters()` 时，PyTorch 递归遍历所有子模块收集参数——这就是为什么你需要把 `nn.Linear` 赋值给 `self.linear` 而非局部变量。局部变量不会出现在 `_modules` 中，不会被 `.parameters()` 遍历到，也不会被 `model.to(device)` 移动。

`forward()` 的调用不需要显式——`model(x)` 会触发 `__call__`，它内部调用 `forward()` 并附加 hook、JIT 编译等前后处理。因此不要直接调用 `model.forward(x)`，使用 `model(x)`。

**核心原则**：`__init__` 定义"有哪些可学习的部分"（参数层），`forward()` 定义"这些部分如何组合"（计算流）。分离使得框架可以自动管理参数、设备和梯度追踪，无需手动编写。


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# === 方式 1: Sequential（简单堆叠）===
model_seq = nn.Sequential(  # 顺序容器
    nn.Linear(784, 256),  # 全连接层 y=Wx+b
    nn.ReLU(),  # ReLU 激活函数
    nn.Dropout(0.2),  # 随机丢弃神经元
    nn.Linear(256, 128),  # 全连接层 y=Wx+b
    nn.ReLU(),  # ReLU 激活函数
    nn.Linear(128, 10)  # 全连接层 y=Wx+b
)
print("Sequential model:")
print(model_seq)

# 查看参数
total_params = sum(p.numel() for p in model_seq.parameters())
trainable_params = sum(p.numel() for p in model_seq.parameters() if p.requires_grad)
print(f"总参数: {total_params:,} (可训练: {trainable_params:,})")

# 前向传播
x_dummy = torch.randn(4, 784)  # batch=4, features=784
out = model_seq(x_dummy)
print(f"输入: {x_dummy.shape} → 输出: {out.shape}")


In [ ]:
# === 方式 2: 自定义 nn.Module（推荐，更灵活）===
import torch.nn as nn
class MNISTClassifier(nn.Module):  # 所有网络层的基类
    def __init__(self, input_dim=784, hidden_dims=[256, 128], num_classes=10, dropout=0.2):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),  # 全连接层 y=Wx+b
                nn.ReLU(),  # ReLU 激活函数
                nn.BatchNorm1d(h_dim),
                nn.Dropout(dropout)  # 随机丢弃神经元
            ])
            prev_dim = h_dim
        
        layers.append(nn.Linear(prev_dim, num_classes))  # 全连接层 y=Wx+b
        self.network = nn.Sequential(*layers)  # 顺序容器
    
    def forward(self, x):
        return self.network(x)  # 返回结果

model = MNISTClassifier()
print("Custom model:")
print(model)

# 参数统计
for name, param in model.named_parameters():
    print(f"  {name:30s}: {list(param.shape)}")


## 5.4 Dataset 与 DataLoader

PyTorch 的数据加载分为两层抽象：

- **`Dataset`**：定义"如何读取一个样本"。实现 `__len__` 和 `__getitem__`
- **`DataLoader`**：包装 Dataset，提供批处理、shuffle、多进程加载

### 内置数据集

`torchvision.datasets` 提供 MNIST、CIFAR-10、ImageNet 等常用数据集，自动下载。


### Dataset 与 DataLoader 的底层协议

PyTorch 的数据加载基于两个协议的配合：

**`Dataset`：定义"如何获取一个样本"**

只需实现两个方法：

```python
class MyDataset(Dataset):
    def __len__(self):
        return len(self.data)    # DataLoader 用它规划 epoch 边界

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]  # 返回 (input, target) 元组
```

`__getitem__` 被 DataLoader 的每个 worker 进程独立调用，因此它必须是**无状态的**——不能依赖共享的全局变量或文件句柄。数据预处理（resize、归一化、augmentation）应在 `__getitem__` 中完成，而非预先处理整个数据集存入内存。

**`DataLoader`：定义"如何组装成一个 batch"**

`DataLoader` 接受一个 `Dataset`，负责：
1. **采样**：根据 `sampler` 决定每个 batch 取哪些索引（随机/顺序/自定义）
2. **组装**：通过 `collate_fn` 将多个 `__getitem__` 返回的样本堆叠为 batch tensor。默认行为是沿第 0 维 stack——因此要求每个样本的 shape 一致
3. **并行**：`num_workers > 0` 时启动子进程并行调用 `__getitem__`——每个 worker 持有独立的 `Dataset` 副本和 Python 解释器

`collate_fn` 是自定义 batching 的入口。当样本变长（如文本序列、边界框数量不等的图像）需要 padding 时，传入自定义 collate 函数：

```python
def pad_collate(batch):
    inputs, targets = zip(*batch)
    inputs = pad_sequence(inputs, batch_first=True)
    return inputs, torch.tensor(targets)

loader = DataLoader(dataset, batch_size=32, collate_fn=pad_collate)
```

> `num_workers` 并非越大越好——每个 worker 启动一个独立进程，进程间通信有开销。对于 I/O 密集型（大量文件读取）设为 4–8 有效；对于内存数据集（数据已在内存中），`num_workers=0` 反而最快。


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

# === 加载 MNIST（torchvision 内置）===
transform = transforms.Compose([
    transforms.ToTensor(),                      # PIL Image → Tensor [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST 统计量标准化
])

train_dataset = datasets.MNIST(
    root='../data',
    train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root='../data',
    train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=0)

print(f"训练集: {len(train_dataset)} 张")
print(f"测试集: {len(test_dataset)} 张")
print(f"每批次: {len(train_loader)} 批 × 64 = {len(train_loader)*64}")

# 查看一个批次
images, labels = next(iter(train_loader))
print(f"图像 shape: {images.shape}")  # [64, 1, 28, 28]
print(f"标签: {labels[:10]}")


In [ ]:
# === 自定义 Dataset 示例 ===
import torch
class SyntheticDataset(Dataset):
    """生成合成回归数据"""
    def __init__(self, n_samples=1000, n_features=10, noise=0.1):
        self.X = torch.randn(n_samples, n_features)  # 标准正态分布随机张量
        true_w = torch.randn(n_features)  # 标准正态分布随机张量
        self.y = self.X @ true_w + noise * torch.randn(n_samples)  # 标准正态分布随机张量
    
    def __len__(self):
        return len(self.X)  # 返回结果
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]  # 返回结果

synth_data = SyntheticDataset(n_samples=500)
synth_loader = DataLoader(synth_data, batch_size=32, shuffle=True)

x_batch, y_batch = next(iter(synth_loader))
print(f"Batch: x={x_batch.shape}, y={y_batch.shape}")


## 5.5 训练循环：每一行为什么在那里

所有深度学习模型的训练循环结构完全一样——变的是模型架构和数据，不变的是下面这五行代码：

```python
for epoch in range(num_epochs):
    for x, y in train_loader:
        optimizer.zero_grad()   # ① 清零上一轮的梯度
        loss = criterion(model(x), y)  # ② 前向传播 + 计算损失
        loss.backward()         # ③ 反向传播，计算所有参数的梯度
        optimizer.step()        # ④ 用梯度更新参数
```

### 逐行分解

**① `optimizer.zero_grad()`** — 清零梯度缓存。

PyTorch 的 `.backward()` 将计算出的梯度**累加**到 `.grad` 属性上，而不是覆盖。如果不先清零，本轮梯度会和上轮残留的梯度叠加，导致参数更新步长翻倍——训练发散。先清零确保每轮从干净的基线开始。

唯一的例外是**梯度累积**：当 GPU 显存装不下大 batch 时，将 batch 拆分为多次小 forward，梯度累加后一次性更新——这时在几个小 forward 之间不清零。

**② `criterion(model(x), y)`** — 计算损失。

`model(x)` 执行前向传播：输入 → 各层逐层计算 → 输出。`criterion` 比较输出和标签，返回一个标量损失值。PyTorch 在这一步同时构建了计算图——每个操作被记录为图的一个节点，反向传播时遍历这些节点。

**③ `loss.backward()`** — 反向传播。

从损失标量开始，逆序遍历计算图，对每个节点应用链式法则，计算该节点的局部梯度并传给上游。所有 `requires_grad=True` 的叶子 Tensor（即模型参数）的 `.grad` 此时被填充。

这一步不改变任何参数的值——只计算梯度。参数更新在下一步发生。

**④ `optimizer.step()`** — 参数更新。

优化器遍历所有参数，对每个参数执行 $\theta \leftarrow \theta - \eta \cdot \nabla \mathcal{L}(\theta)$（Adam 等优化器有更复杂的更新规则，但本质相同）。这一步完成后，模型稍微变好了一点点。

### 评估循环：不更新参数

```python
model.eval()                    # 切换为评估模式
with torch.no_grad():           # 不构建计算图
    for x, y in test_loader:
        pred = model(x)
        # 计算准确率等指标
```

`model.eval()` 告诉 BatchNorm 用全局统计量替代 batch 统计量，告诉 Dropout 不要随机置零。`torch.no_grad()` 关闭梯度追踪——不构建计算图，节省显存和计算时间。


In [ ]:
# === 完整训练循环：MNIST MLP with PyTorch ===
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 模型、损失函数、优化器
model = MNISTClassifier(input_dim=784, hidden_dims=[128, 64], num_classes=10).to(device)  # 将数据移到 GPU 或 CPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()  # 切换到训练模式（启用 Dropout/BatchNorm）
    total_loss, correct = 0, 0
    for data, target in loader:
        data = data.view(data.size(0), -1).to(device)  # flatten 28x28 → 784
        target = target.to(device)  # 将数据移到 GPU 或 CPU
        
        optimizer.zero_grad()  # 清空上一轮的梯度
        output = model(data)
        loss = criterion(output, target)
        loss.backward()  # 反向传播，计算梯度
        optimizer.step()  # 用累积的梯度更新参数
        
        total_loss += loss.item()  # 取出单个数值
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()  # 沿指定维度求和
    
    return total_loss / len(loader), correct / len(loader.dataset)  # 返回结果

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()  # 切换到评估模式（禁用 Dropout/BatchNorm）
    total_loss, correct = 0, 0
    for data, target in loader:
        data = data.view(data.size(0), -1).to(device)  # 将数据移到 GPU 或 CPU
        target = target.to(device)  # 将数据移到 GPU 或 CPU
        output = model(data)
        total_loss += criterion(output, target).item()  # 取出单个数值
        correct += (output.argmax(1) == target).sum().item()  # 沿指定维度求和
    return total_loss / len(loader), correct / len(loader.dataset)  # 返回结果

# 训练 5 epoch
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
for epoch in range(5):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Acc={train_acc:.3%} | "
          f"Val Loss={val_loss:.4f}, Acc={val_acc:.3%}")

print(f"\n✅ 测试集最终准确率: {val_acc:.2%}")


## 5.6 模型保存与加载：两种方式的选择

### 方式一：保存整个模型（简单但受限）

```python
torch.save(model, 'model.pt')
model = torch.load('model.pt')
```

将整个模型对象序列化到文件。优点是方便——一行保存一行加载。缺点是在不同环境中加载可能失败——Python 版本不同、PyTorch 版本不同、模型类定义不在当前命名空间中时都会报错。

### 方式二：只保存参数（推荐）

```python
torch.save(model.state_dict(), 'model.pt')     # 保存
model = MyModel()
model.load_state_dict(torch.load('model.pt'))   # 加载
```

`state_dict` 是一个 Python 字典，键是参数名（如 `'fc1.weight'`），值是参数的 Tensor。只保存数值不保存模型结构。加载时需要重新实例化模型，然后 `load_state_dict` 将数值填回去。

优点：跨环境兼容（只要模型类的定义相同）、文件更小、可以只加载部分参数（迁移学习场景——加载预训练权重但跳过分类头）。

### 保存 checkpoint（训练中断恢复）

```python
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch,
    'loss': loss,
}
torch.save(checkpoint, 'checkpoint.pt')

# 恢复训练
checkpoint = torch.load('checkpoint.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1
```

同时保存模型参数、优化器状态（Adam 的动量缓存）、训练进度——可以从中断点精确恢复，不需要重新训练。


In [ ]:
# === 保存模型 ===
import torch
save_path = './models/mnist_mlp.pth'
os.makedirs('./models', exist_ok=True)

# 保存 state_dict（推荐）
torch.save({  # 保存模型/张量到文件
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': 5,
    'val_acc': val_acc,
}, save_path)
print(f"模型保存至: {save_path}")

# === 加载模型 ===
loaded_model = MNISTClassifier(input_dim=784, hidden_dims=[128, 64])
checkpoint = torch.load(save_path, map_location=device)  # 从文件加载模型/张量
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.to(device)  # 将数据移到 GPU 或 CPU

# 验证加载的模型
loaded_val_loss, loaded_val_acc = evaluate(loaded_model, test_loader, criterion, device)
print(f"恢复模型验证准确率: {loaded_val_acc:.3%} (原始: {val_acc:.3%})")

# 清理
import os as _os
_os.remove(save_path)


## 本章小结

| 组件 | PyTorch 实现 |
|------|-------------|
| 多维数组 | `torch.Tensor` (CPU/GPU) |
| 自动求导 | `.backward()` + `requires_grad=True` |
| 网络层 | `nn.Linear`, `nn.Conv2d`, `nn.ReLU`... |
| 模型容器 | `nn.Sequential` / 自定义 `nn.Module` |
| 损失函数 | `nn.MSELoss`, `nn.CrossEntropyLoss`... |
| 优化器 | `optim.SGD (Stochastic Gradient Descent)`, `optim.Adam`... |
| 数据加载 | `Dataset` + `DataLoader` |
| 模型保存 | `torch.save(model.state_dict(), ...)` |

✅ 第五章完成！下一章将用 PyTorch 在 CIFAR-10 上构建卷积神经网络。
